<a href="https://colab.research.google.com/github/Subparplatinum/Moss-Species-Classifier/blob/main/moss_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Moss classification model creation

In [1]:
import kagglehub
path = kagglehub.dataset_download("alexanderuzhinskiy/moss-species-classification-dataset")


100%|██████████| 12.1M/12.1M [00:00<00:00, 57.6MB/s]

Extracting files...


In [2]:
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import random_split

transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

batch_size = 4

classes = ('abietinella_abietina', 'hylocomium_splendens', 'hypnum_cupressiforme',
           'pleurozium_schreberi', 'pseudoscleropodium_purum')

# Load the entire dataset from the main path, assuming species folders are directly under it
full_dataset = torchvision.datasets.ImageFolder(root="/root/.cache/kagglehub/datasets/alexanderuzhinskiy/moss-species-classification-dataset/versions/1/mosses",
                                                transform=transform)

# Get the number of classes from the full dataset
classes = tuple(full_dataset.classes)
print(classes)

# Define the split ratios (95% for training, 5% for testing)
train_size = int(0.95 * len(full_dataset))
test_size = len(full_dataset) - train_size

# Split the dataset into training and testing sets
trainset, testset = random_split(full_dataset, [train_size, test_size])

# Create DataLoaders for the training and testing sets
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

('abietinella_abietina', 'hylocomium_splendens', 'hypnum_cupressiforme', 'pleurozium_schreberi', 'pseudoscleropodium_purum')


In [3]:
import torch.nn as nn
import torch.nn.functional as F

# Define the device
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        # Create a dummy input to pass through the conv layers to determine output size

        with torch.no_grad():
            dummy_input = torch.zeros(1, 3, 256, 256) # Files are 256 x 256
            x = self.pool(F.relu(self.conv1(dummy_input)))
            x = self.pool(F.relu(self.conv2(x)))
            self._to_linear = torch.flatten(x, 1).shape[1]

        self.fc1 = nn.Linear(self._to_linear, 128)
        self.fc2 = nn.Linear(128, 32)
        self.fc3 = nn.Linear(32, len(classes))

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1) # flatten all dimensions except batch

        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x


net = Net()
net.to(device) # Move the model to the GPU

import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

cpu


In [4]:
for epoch in range(30):  # loop over the dataset multiple times

    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        # get the inputs; data is a list of [inputs, labels]
        inputs, labels = data[0].to(device), data[1].to(device) # Move inputs and labels to GPU

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
        if i % 100 == 99:    # print every 100 mini-batches
          print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.3f}')
          running_loss = 0.0

print('Finished Training')

torch.save(net.state_dict(), "/root/moss_classifier.pth")

[1,   100] loss: 0.079
[2,   100] loss: 0.078
[3,   100] loss: 0.075
[4,   100] loss: 0.072
[5,   100] loss: 0.071
[6,   100] loss: 0.069
[7,   100] loss: 0.065
[8,   100] loss: 0.061
[9,   100] loss: 0.057
[10,   100] loss: 0.042
[11,   100] loss: 0.023
[12,   100] loss: 0.010
[13,   100] loss: 0.007
[14,   100] loss: 0.002
[15,   100] loss: 0.000
[16,   100] loss: 0.000
[17,   100] loss: 0.000
[18,   100] loss: 0.000
[19,   100] loss: 0.000
[20,   100] loss: 0.000
[21,   100] loss: 0.000
[22,   100] loss: 0.000
[23,   100] loss: 0.000
[24,   100] loss: 0.000
[25,   100] loss: 0.000
[26,   100] loss: 0.000
[27,   100] loss: 0.000
[28,   100] loss: 0.000
[29,   100] loss: 0.000
[30,   100] loss: 0.000
Finished Training


# Test

In [6]:
net = Net()
net.load_state_dict(torch.load("/root/moss_classifier.pth", weights_only=True))

correct = 0
total = 0
# since we're not training, we don't need to calculate the gradients for our outputs
with torch.no_grad():
    for data in testloader:
        images, labels = data[0].to(device), data[1].to(device) # Move images and labels to GPU
        # calculate outputs by running images through the network
        outputs = net(images)
        # the class with the highest energy is what we choose as prediction
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy of the network on the {total} test images: {100 * correct // total} %')

Accuracy of the network on the 30 test images: 50 %
